# Plotly #

In [31]:
# Import Libraries

# Data Management
import pandas as pd
import numpy as np

# Plots
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Handle Files
import sys
import os

# Import Local Functions
sys.path.append(os.path.abspath("../source"))
from config import get_tickers
from data_downloader import get_market_data

In [2]:
# Import a portfolio
bab_portfolio = pd.read_csv(r'../additional_data/betting_against_beta_portfolio.csv')
bab_portfolio.set_index('Date', inplace=True)
bab_portfolio.index = pd.to_datetime(bab_portfolio.index)

bab_portfolio

,portfolio_returns
Date,
2000-01-03,-0.015830
2000-01-04,0.010408
2000-01-05,0.018127
2000-01-06,0.023416
2000-01-07,-0.003012
...,...
2024-12-24,0.002694
2024-12-26,-0.000779
2024-12-27,0.002842


### Waterfall Charts ###

In [3]:
# Create Annual Returns
bab_annual_returns = bab_portfolio.groupby(bab_portfolio.index.year).mean()
bab_annual_returns = bab_annual_returns * 252

bab_annual_returns

,portfolio_returns
Date,
2000,0.463768
2001,0.121398
2002,0.157834
2003,0.072124
2004,0.171588
2005,0.024449
2006,0.107448
2007,-0.058623
2008,0.049741


In [4]:
# Create a Waterfall Chart
fig = go.Figure(go.Waterfall(
    name="Annual Returns",                                                          # Name of the variable
    orientation="v",                                                                # Vertical
    measure=["relative"] * len(bab_annual_returns) + ["total"],                     # How to interpret each bar (includes the total)
    x=bab_annual_returns.index.astype(str).tolist() + ["Cumulative"],               # Years in X axis (includes the total)
    y=bab_annual_returns["portfolio_returns"].tolist() + [0],                       # Values of each Bar
    text=[f"{v:.1f}%" for v in bab_annual_returns["portfolio_returns"]] + [""],     # Text of each Bar
    textposition="outside",                                                         # Position of Text
    connector={"line": {"color":"rgba(63, 63, 63, 0.7)"}}                           # Lines that connect the vars
))

fig.update_layout(
    title="Betting Against Beta Annual Portfolio Performance (Waterfall)",
    yaxis_title="Return (%)",
    waterfallgap=0.4                                                                # Spaces between gaps
)

fig.show()

### Sunburst ###

In [5]:
tickers = get_tickers("7.3")

tickers

['AAPL', 'MSFT', 'JPM', 'GS', 'JNJ', 'PFE', 'XOM', 'CVX', 'AMZN', 'TSLA']

In [6]:
# DataFrame to store everything
df_prices = pd.DataFrame()

for ticker in tickers:
    df = get_market_data(
        ticker=ticker, 
        start_date='2015-01-01', 
        end_date='2025-01-01', 
        returns=False
    )
    
    prices = df['close'].rename(ticker)
    
    df_prices = pd.concat([df_prices, prices], axis=1)

In [7]:
df_prices.index = pd.to_datetime(df_prices.index)

df_prices

,AAPL,MSFT,JPM,GS,JNJ,PFE,XOM,CVX,AMZN,TSLA
2015-01-02,24.261049,39.933060,46.948086,157.149063,77.444176,19.017998,58.423470,70.993698,15.426000,14.620667
2015-01-05,23.577570,39.565842,45.490574,152.242432,76.903267,18.914801,56.824894,68.155960,15.109500,14.006000
2015-01-06,23.579792,38.985115,44.311054,149.162659,76.525406,19.072632,56.522812,68.124413,14.764500,14.085333
2015-01-07,23.910437,39.480438,44.378677,151.385590,78.214767,19.333649,57.095524,68.067665,14.921000,14.063333
2015-01-08,24.829130,40.641865,45.370380,153.802551,78.829735,19.728216,58.045860,69.625252,15.023000,14.041333
...,...,...,...,...,...,...,...,...,...,...
2024-12-24,257.286682,436.929108,238.440521,573.965576,142.416565,25.374838,103.530914,139.002731,229.050003,462.279999
2024-12-26,258.103729,435.715790,239.257263,572.429138,142.152908,25.204027,103.618484,139.138031,227.050003,454.130005
2024-12-27,254.685867,428.177216,237.318726,567.455688,141.635391,25.260965,103.608757,139.157364,223.750000,431.660004
2024-12-30,251.307877,422.508362,235.498260,564.865479,139.965652,25.071177,102.908165,138.258636,221.300003,417.410004


In [8]:
# Imagine you want to create a portfolio now
df_prices_now = df_prices.iloc[-1]
df_prices_now.name = 'current_prices'

df_prices_now

AAPL    249.534180
MSFT    419.196564
JPM     235.882050
GS      563.949585
JNJ     141.215500
PFE      25.175560
XOM     104.669357
CVX     139.969101
AMZN    219.389999
TSLA    403.839996
Name: current_prices, dtype: float64

In [9]:
ticker_to_sector = {
    "AAPL": "Technology",
    "MSFT": "Technology",
    "JPM": "Financials",
    "GS": "Financials",
    "JNJ": "Healthcare",
    "PFE": "Healthcare",
    "XOM": "Energy",
    "CVX": "Energy",
    "AMZN": "Consumer Discretionary",
    "TSLA": "Consumer Discretionary"
}

In [10]:
sectors = pd.Series(ticker_to_sector)
sectors.name = "Sector"

sectors

AAPL                Technology
MSFT                Technology
JPM                 Financials
GS                  Financials
JNJ                 Healthcare
PFE                 Healthcare
XOM                     Energy
CVX                     Energy
AMZN    Consumer Discretionary
TSLA    Consumer Discretionary
Name: Sector, dtype: object

In [11]:
df_weights = pd.DataFrame({
    "stocks": df_prices_now.index,
    "prices": df_prices_now,
    "sector": sectors
})

df_weights['prices'].astype(float)
df_weights['weights'] = (df_weights['prices'] / df_weights['prices'].sum()) * 100

df_weights

,stocks,prices,sector,weights
AAPL,AAPL,249.534180,Technology,9.970113
MSFT,MSFT,419.196564,Technology,16.748957
JPM,JPM,235.882050,Financials,9.424644
GS,GS,563.949585,Financials,22.532550
JNJ,JNJ,141.215500,Healthcare,5.642251
PFE,PFE,25.175560,Healthcare,1.005887
XOM,XOM,104.669357,Energy,4.182054
CVX,CVX,139.969101,Energy,5.592452
AMZN,AMZN,219.389999,Consumer Discretionary,8.765706
TSLA,TSLA,403.839996,Consumer Discretionary,16.135387


In [12]:
# Sunburst Plot
fig = px.sunburst(
    df_weights,                                                 # Data of Use
    path=['sector', 'stocks'],                                  # Classification Order
    values='weights',                                           # Values to Plot
    color='prices',                                             # Variables for Color
    color_continuous_scale='RdBu',                              # Scale of Colors
    color_continuous_midpoint = df_weights['prices'].mean(),    # Midpoint for Colors
)

fig.update_layout(
    title="Weights Sunburst by Sector",
    margin=dict(t=50, l=0, r=0, b=0),                           # Margins
)

fig.show()

### Heatmaps ###

In [13]:
# Calculate the Daily Weights
df_daily_weights = df_prices.div(df_prices.sum(axis=1), axis=0)
df_daily_weights = df_daily_weights.loc['2023':]

df_daily_weights

,AAPL,MSFT,JPM,GS,JNJ,PFE,XOM,CVX,AMZN,TSLA
2023-01-03,0.084472,0.160563,0.086371,0.221016,0.112114,0.030035,0.066377,0.106232,0.058780,0.074040
2023-01-04,0.085468,0.153762,0.087303,0.222283,0.113500,0.029415,0.066666,0.105255,0.058399,0.077948
2023-01-05,0.085362,0.150617,0.088110,0.221845,0.113728,0.029415,0.068803,0.108165,0.057553,0.076401
2023-01-06,0.087054,0.149899,0.088326,0.220956,0.112774,0.029668,0.068495,0.107196,0.058628,0.077003
2023-01-09,0.087105,0.150829,0.087654,0.223297,0.109469,0.028096,0.066984,0.105987,0.059292,0.081288
...,...,...,...,...,...,...,...,...,...,...
2024-12-24,0.098642,0.167516,0.091417,0.220055,0.054602,0.009729,0.039693,0.053293,0.087817,0.177236
2024-12-26,0.099393,0.167790,0.092135,0.220436,0.054742,0.009706,0.039902,0.053581,0.087435,0.174881
2024-12-27,0.099771,0.167734,0.092967,0.222295,0.055484,0.009896,0.040588,0.054514,0.087652,0.169099
2024-12-30,0.099761,0.167722,0.093485,0.224234,0.055562,0.009952,0.040851,0.054884,0.087849,0.165698


In [14]:
# Calculate the average for the color palette
avg_weights = df_daily_weights.mean().sort_values(ascending=False)
ordered_stocks = avg_weights.index
df_ordered = df_daily_weights[ordered_stocks]

In [15]:
# Plot
z = df_ordered.T.values

fig = go.Figure(data=go.Heatmap(
    z=z,                                # Data Values
    x=df_ordered.index,                 # Dates for the X-Axis
    y=df_ordered.columns,               # Stocks for the Y-AXis
    colorscale='Viridis',               # Palette
    zmin=z.min(),                       # Minimum Value for the Palette
    zmax=z.max(),                       # Maximum Value for the Palette
    colorbar=dict(title="Weight")
))

fig.update_layout(
    title="Evolution of Portfolio Weights Over Time",
    xaxis_title="Date",
    yaxis_title="Stocks",
    xaxis=dict(
        tickmode='array',
        tickvals=df_ordered.index[::20], # Show dates each 20 days (adjust depending on your data)
        ticktext=[d.strftime('%Y-%m-%d') for d in df_ordered.index[::20]]
    )
)

fig.show()

### Stacked Area Charts ###

In [16]:
# Stacked Area Charts
df_plot = df_daily_weights.copy()
df_plot["date"] = df_plot.index

# Plot
fig = px.area(
    df_plot,                                # Data
    x="date",                               # X-Axis
    y=df_plot.columns,                      # Columns: the variables of use
    title="Portfolio Weights Over Time",    # Title
    groupnorm="percent"                     # Ensure the weights sum 100%
)

fig.update_layout(
    yaxis_title="Portfolio Weight (%)",
    xaxis_title="Date",
    legend_title="Stocks"
)

fig.show()

### Radar Chart ###

In [17]:
# Calculate Returns
returns = np.log(df_prices) - np.log(df_prices.shift(1))
returns.dropna(inplace=True)
returns

,AAPL,MSFT,JPM,GS,JNJ,PFE,XOM,CVX,AMZN,TSLA
2015-01-05,-0.028576,-0.009238,-0.031537,-0.031721,-0.007009,-0.005441,-0.027743,-0.040793,-0.020731,-0.042950
2015-01-06,0.000094,-0.014786,-0.026271,-0.020437,-0.004926,0.008310,-0.005330,-0.000463,-0.023098,0.005648
2015-01-07,0.013925,0.012625,0.001525,0.014793,0.021836,0.013593,0.010081,-0.000833,0.010544,-0.001563
2015-01-08,0.037703,0.028993,0.022100,0.015839,0.007832,0.020203,0.016508,0.022625,0.006813,-0.001566
2015-01-09,0.001072,-0.008441,-0.017540,-0.015466,-0.013723,0.004604,-0.001410,-0.020127,-0.011818,-0.018981
...,...,...,...,...,...,...,...,...,...,...
2024-12-24,0.011413,0.009330,0.016310,0.020823,0.003985,0.001123,0.000940,0.006067,0.017573,0.070991
2024-12-26,0.003171,-0.002781,0.003419,-0.002680,-0.001853,-0.006754,0.000845,0.000973,-0.008770,-0.017787
2024-12-27,-0.013331,-0.017453,-0.008135,-0.008726,-0.003647,0.002257,-0.000094,0.000139,-0.014641,-0.050745
2024-12-30,-0.013352,-0.013328,-0.007701,-0.004575,-0.011859,-0.007541,-0.006785,-0.006479,-0.011010,-0.033569


In [18]:
# Calculate Metrics
metrics = {}

for stock in returns.columns:
    r = returns[stock]

    # Annualized Return
    mean_daily = r.mean()
    ann_return = (1 + mean_daily)**252 - 1

    # Annualized Volatility
    ann_vol = r.std() * np.sqrt(252)

    # Sharpe Ratio (rf=0)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    # Maximum Drawdown
    cum_ret = (1 + r).cumprod()
    rolling_max = cum_ret.cummax()
    drawdown = cum_ret / rolling_max - 1
    max_dd = drawdown.min()

    # VaR 95%
    var_95 = np.percentile(r, 5)

    # Semi-Deviation
    downside = r[r < 0]
    semidev = downside.std() * np.sqrt(252)

    metrics[stock] = {
        "Annual Return": ann_return,
        "Annual Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd,
        "VaR (95%)": var_95,
        "Semi-Deviation": semidev
    }

df_metrics = pd.DataFrame(metrics).T

In [19]:
stocks_for_use = df_metrics.loc[['AAPL', 'PFE', 'TSLA']]

stocks_for_use

,Annual Return,Annual Volatility,Sharpe Ratio,Max Drawdown,VaR (95%),Semi-Deviation
AAPL,0.262921,0.284654,0.923654,-0.401614,-0.027362,0.212249
PFE,0.028502,0.231928,0.122890,-0.579951,-0.021597,0.160826
TSLA,0.394169,0.570053,0.691461,-0.798813,-0.052740,0.411393


In [20]:
# Normalize Data
df_norm = (stocks_for_use - stocks_for_use.min().min()) / (stocks_for_use.max().max() - stocks_for_use.min().min())
df_norm.index.name = 'Stock'
df_norm.reset_index(inplace=True)
df_norm

,Stock,Annual Return,Annual Volatility,Sharpe Ratio,Max Drawdown,VaR (95%),Semi-Deviation
0,AAPL,0.616403,0.629020,1.000000,0.230598,0.447876,0.586985
1,PFE,0.480308,0.598409,0.535106,0.127063,0.451222,0.557131
2,TSLA,0.692601,0.794712,0.865198,0.000000,0.433142,0.702600


In [21]:
# Long format
df_long = df_norm.melt(id_vars='Stock', var_name="Metric", value_name="Value")

df_long

,Stock,Metric,Value
0,AAPL,Annual Return,0.616403
1,PFE,Annual Return,0.480308
2,TSLA,Annual Return,0.692601
3,AAPL,Annual Volatility,0.629020
4,PFE,Annual Volatility,0.598409
5,TSLA,Annual Volatility,0.794712
6,AAPL,Sharpe Ratio,1.000000
7,PFE,Sharpe Ratio,0.535106
8,TSLA,Sharpe Ratio,0.865198
9,AAPL,Max Drawdown,0.230598


In [22]:
# Radar chart
fig = px.line_polar(
    df_long,
    r="Value",
    theta="Metric",
    color="Stock",
    line_close=True,
    title="Stocks Risk Metrics Radar Chart"
)

fig.update_traces(fill="toself")
fig.show()

### Candle Sticks ###

In [23]:
# Download Data
stock = 'NVDA'

df = get_market_data(
        ticker=stock, 
        start_date='2023-01-01', 
        end_date='2025-01-01', 
        returns=False
)

df.index.name = 'date'
df.index = pd.to_datetime(df.index)

In [24]:
df

Price,close,high,low,open,volume
date,,,,,
2023-01-03,14.301478,14.981836,14.082685,14.836972,401277000
2023-01-04,14.735070,14.838972,14.227551,14.553243,431324000
2023-01-05,14.251528,14.550245,14.134638,14.477314,389168000
2023-01-06,14.844964,14.995821,14.020744,14.460327,405044000
2023-01-09,15.613239,16.040835,15.126699,15.269564,504231000
...,...,...,...,...,...
2024-12-24,140.189468,141.869095,138.619803,139.969515,105157000
2024-12-26,139.899521,140.819334,137.700003,139.669575,116205600
2024-12-27,136.980164,138.989736,134.680677,138.519837,170582600


In [30]:
# Candlesticks
fig = go.Figure(
    data=[go.Candlestick(
        x=df.index,             # X-Axis
        open=df['open'],        # Open Price
        high=df['high'],        # High Price
        low=df['low'],          # Low Price
        close=df['close'],      # Close Price.
        
    )]
)

fig.update_layout(
    yaxis_title=f"{stock} Prices",
    xaxis_title="Date",
    title=dict(
        text=f'{stock} Prices'
        ),
    legend_title="Stocks"
)

# True if you want to see the Range Slider
fig.update_layout(xaxis_rangeslider_visible=False)

fig.show()

In [38]:
# Adding Volume
fig = make_subplots(
            rows=2,                     # Number of Rows               
            cols=1,                     # Number of Columns
            shared_xaxes=True,          # Share X-Axis (True if Dates)
            vertical_spacing=0.015,     # Vertical Space between Charts
            row_heights=[0.7, 0.3],     # Size of Charts   
            shared_yaxes=True           # Share Y-Axis (if necessary)
        )

fig.add_trace(
    go.Candlestick(
        x=df.index,                     # Dates
        open=df['open'],                # Open Price
        high=df['high'],                # High Price
        low=df['low'],                  # Low Price
        close=df['close'],              # Close Price
        name='Candles'
    ),
    row=1,                              # Subplot
    col=1
)

fig.add_trace(
    go.Bar(                             # Bar Chart
        x=df.index,                     # Dates
        y=df['volume'],                 # Volume Data
        marker=dict(
            color="blue"                # Bar Colors
        ),
        name='Volume',                  # Label
        showlegend=False,               # Show Legends
    ),
    row=2,                              # Subplot
    col=1
)

fig.update_xaxes(
    title_text="Date",                  # X-Label
    title_font=dict(size=12),           # Letter Size
    row=2,                              # Subplot
    col=1
)

fig.update_yaxes(
    title_text="Price",                 # Y-Label
    title_font=dict(size=12),           # Letter Size
    row=1,                              # Subplot
    col=1
)


fig.update_yaxes(
    title_text="Volume",                # Y-Label
    title_font=dict(size=12),           # Letter Size
    row=2,                              # Subplot
    col=1
)

fig.update_layout(
    xaxis_rangeslider_visible=False     # True if you want to see the Range Slider
)

fig.update_layout(
        height=600                      # Plot Height
)

# Title Configuration
fig.update_layout(
        title={
            'text': f'{stock} Candlesticks with Volume',
            'font': {
                'size': 25
            }
        },
        title_x=0.5,
        title_xanchor="center"
    )

fig.show()

### Bubble Chart ###

In [44]:
# Create Data
portfolio_dataframe = pd.DataFrame()
portfolio_dataframe['AAPL'] = returns['AAPL']
portfolio_dataframe['TSLA'] = returns['TSLA']
portfolio_dataframe.index.name = 'date'

# Create an Equal-Weighted Portfolio
portfolio_dataframe['EWP'] = (1/2 * portfolio_dataframe['AAPL']) + (1/2 * portfolio_dataframe['TSLA'])

portfolio_dataframe

,AAPL,TSLA,EWP
date,,,
2015-01-05,-0.028576,-0.042950,-0.035763
2015-01-06,0.000094,0.005648,0.002871
2015-01-07,0.013925,-0.001563,0.006181
2015-01-08,0.037703,-0.001566,0.018068
2015-01-09,0.001072,-0.018981,-0.008954
...,...,...,...
2024-12-24,0.011413,0.070991,0.041202
2024-12-26,0.003171,-0.017787,-0.007308
2024-12-27,-0.013331,-0.050745,-0.032038


In [53]:
# Create Rolling Volatility
rolling_vol_df = portfolio_dataframe.rolling(window=252).var()
rolling_vol_df.dropna(inplace=True)

rolling_vol_df

,AAPL,TSLA,EWP
date,,,
2016-01-04,0.000284,0.000616,0.000286
2016-01-05,0.000283,0.000609,0.000281
2016-01-06,0.000284,0.000610,0.000283
2016-01-07,0.000291,0.000611,0.000286
2016-01-08,0.000285,0.000613,0.000285
...,...,...,...
2024-12-24,0.000201,0.001535,0.000525
2024-12-26,0.000201,0.001536,0.000525
2024-12-27,0.000202,0.001546,0.000530


In [55]:
fig = px.scatter(
    rolling_vol_df.reset_index(),                   # Data
    x="AAPL",                                       # X-Axis
    y="TSLA",                                       # Y-Axis
    size="EWP",                                     # Bubble Size
    color='EWP',                                    # Bubble Color 
    hover_name=rolling_vol_df.index.astype(str),    # Hover Names
    size_max=30,                                    # Max Size for Bubbles
    title="Bubble Chart of Rolling Variances"
)

fig.update_layout(
    xaxis_title="AAPL Rolling Variance",
    yaxis_title="TSLA Rolling Variance",
    template="plotly_dark",
    coloraxis_colorbar=dict(title="Date")
)

fig.update_layout(height=600)

fig.show()

### Animated Plots ###

In [56]:
# Download Betas
betas_df = pd.read_csv('../additional_data/capm_rbetas.csv')
betas_df.set_index('Date', inplace=True)
betas_df.index = pd.to_datetime(betas_df.index)

betas_df

,A,AAL,AAP,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,...,XEL,XOM,XRAY,XRX,XYL,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,NaN,1.335080,NaN,NaN,1.084290,-0.114335,NaN,1.052979,...,0.348611,0.530261,0.557190,0.828263,NaN,0.530965,NaN,0.748833,0.852891,NaN
2000-01-04,NaN,NaN,NaN,1.418454,NaN,NaN,1.051470,-0.091197,NaN,1.168953,...,0.264786,0.529788,0.506847,0.848708,NaN,0.522835,NaN,0.720079,0.888770,NaN
2000-01-05,NaN,NaN,NaN,1.440879,NaN,NaN,1.060008,-0.084862,NaN,1.189834,...,0.268014,0.525762,0.508524,0.851524,NaN,0.521590,NaN,0.726198,0.893849,NaN
2000-01-06,NaN,NaN,NaN,1.440651,NaN,NaN,1.060747,-0.085251,NaN,1.190678,...,0.267661,0.527185,0.508530,0.851091,NaN,0.521318,NaN,0.725240,0.894127,NaN
2000-01-07,NaN,NaN,NaN,1.447918,NaN,NaN,1.038326,-0.001634,NaN,1.208871,...,0.259260,0.502271,0.486571,0.859092,NaN,0.471773,NaN,0.673694,0.864814,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,0.756693,1.269029,1.006397,0.900453,0.214614,1.303138,0.153382,0.532669,0.639139,1.276747,...,-0.038703,0.198814,0.542305,1.006918,0.985409,0.457147,0.436171,1.235720,1.404490,0.510449
2024-12-26,0.756891,1.270957,1.004937,0.900734,0.215589,1.304253,0.152650,0.532828,0.640070,1.275916,...,-0.038702,0.198456,0.538876,1.006911,0.984227,0.455964,0.435341,1.233641,1.402107,0.509350
2024-12-27,0.747591,1.253666,0.992316,0.906195,0.221473,1.302630,0.153808,0.533355,0.646099,1.264858,...,-0.036025,0.195271,0.531051,0.990680,0.978535,0.456436,0.432578,1.242268,1.397838,0.507575


In [59]:
# Choose Last Dates
last_dates = pd.DatetimeIndex(
    betas_df.index.to_series().groupby(betas_df.index.year).max()
)

# Years
years = [d.year for d in last_dates]

# Betas
betas_last = betas_df.loc[last_dates]
betas_last = betas_last.dropna(axis = 1, how = 'all')
betas_last.index = betas_last.index.year

betas_last

,A,AAL,AAP,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,...,XEL,XOM,XRAY,XRX,XYL,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2000,1.731164,NaN,NaN,1.803924,NaN,NaN,0.182539,0.071956,NaN,2.528820,...,0.100018,0.049818,0.262340,0.880732,NaN,0.674608,NaN,1.071322,0.859159,NaN
2001,1.933123,NaN,NaN,1.611954,NaN,NaN,0.310849,0.180566,NaN,2.157660,...,0.347328,0.497962,0.511553,1.104915,NaN,0.383494,NaN,1.107232,0.502264,NaN
2002,1.373665,NaN,0.644578,1.013274,NaN,NaN,0.710204,0.255607,1.162431,1.479075,...,0.926783,0.934881,0.468421,1.200480,NaN,0.421794,0.404942,0.799893,0.875828,NaN
2003,1.577994,NaN,0.667797,1.361338,NaN,NaN,0.840797,0.373613,0.837332,1.565247,...,0.579566,0.680816,0.871811,1.210453,NaN,0.806426,0.521462,0.843382,0.884245,NaN
2004,1.764981,NaN,0.935145,1.291330,NaN,NaN,0.783147,0.552325,0.998400,1.450715,...,0.614062,0.747777,1.068776,1.238124,NaN,0.872803,1.545044,1.233914,0.818419,NaN
2005,1.130607,NaN,1.144314,1.671347,NaN,NaN,0.601791,0.549483,0.877129,1.297107,...,0.843211,1.424418,0.735839,0.961303,NaN,0.839800,0.684353,0.952593,1.046803,NaN
2006,1.550053,2.212218,0.526205,1.572629,NaN,NaN,0.736149,0.473672,0.905671,1.500047,...,0.554736,0.945440,0.918474,1.048195,NaN,1.263893,0.658969,1.207869,0.812101,NaN
2007,0.986553,1.205219,0.840508,1.244844,NaN,NaN,0.677295,0.730065,0.810955,0.755533,...,0.884028,1.151669,0.519991,1.011760,NaN,0.915382,0.729732,0.674956,1.253725,NaN
2008,0.980290,1.837888,0.918024,0.942234,NaN,NaN,0.502330,0.722217,0.751337,1.117280,...,0.615349,1.050140,0.702857,0.994770,NaN,0.845296,0.697907,0.861606,1.337913,NaN


In [62]:
# Histograms
fig = go.Figure()

# Loop
for year in betas_last.index:
    fig.add_trace(
        go.Histogram(
            x=betas_last.loc[year].values,  # Values of Betas
            name=str(year),                 # Year as Legend
            opacity=0.5                     # Opacity of Bars
        )
    )

fig.update_layout(
    barmode='overlay',                      # Bar Mode (Overlay)
    title='Beta Evolution per Year',
    xaxis_title='Beta',
    yaxis_title='Stock Frequency',
    legend=dict(
        title='Year',                       # Legends Title
        orientation='v',                    # Bar Orientation (v for vertical)
        x=1.05,                             # Horizontal Position
        y=1                                 # Vertical Position
    )
)

fig.show()

In [68]:
# Long Dataframe
df_long = betas_last.reset_index().melt(
    id_vars="Date", 
    var_name="Stock", 
    value_name="Beta"
)

df_long.rename(columns={"Date": "Year"}, inplace=True)

df_long

,Year,Stock,Beta
0,2000,A,1.731164
1,2001,A,1.933123
2,2002,A,1.373665
3,2003,A,1.577994
4,2004,A,1.764981
...,...,...,...
15270,2020,ZTS,0.905951
15271,2021,ZTS,0.803403
15272,2022,ZTS,1.001542
15273,2023,ZTS,1.033415


In [79]:
# Histogram with animation
fig = px.histogram(
    df_long,                                # Data
    x="Beta",                               # Values for Histogram
    animation_frame="Year",                 # Variable for Animation
    nbins=50,                               # Bins of Distribution
    title="Beta Distribution Over Time",    # Title
    opacity=0.5,                            # Opacity of Bars
    color_discrete_sequence=["#2ECC71"]     # Color
)

# Vertical Line
fig.add_vline(
    x=1,                                    # Value for Vertical Line
    line_width=2,                           # Line width
    line_dash="dash",                       # Style (Dashed)
    line_color="black",                     # Line Color
    annotation_text="Beta=1",               # Label
    annotation_position="top"               # Position
)

# Config
fig.update_layout(
    xaxis_title="Beta",
    yaxis_title="Stock Frequency",
    template="plotly_dark"
)

fig.update_layout(height=600)

fig.show()